In [1]:
import csv
import xml.etree.ElementTree as ET

def csv_to_kml(csv_path, kml_path, label_segments=False, base_dir=None):
    """
    Convert a CSV with segment data to a KML file.

    Args:
        csv_path (str): Path to input CSV.
        kml_path (str): Path to output KML.
        label_segments (bool): If True, adds offset labels for each segment (no visible pin icon).
    """

    # Approximate degree offsets at ~44°N
    lat_per_ft = 1 / 364000   # degrees latitude per foot
    lon_per_ft = 1 / 262000   # degrees longitude per foot

    # Line offset (~10 ft apart for visibility)
    lat_offset = 10 * lat_per_ft
    lon_offset = 10 * lon_per_ft

    # Create root KML element
    kml = ET.Element('kml', xmlns='http://www.opengis.net/kml/2.2')
    document = ET.SubElement(kml, 'Document')

    # ---------- Styles ----------
    # KML color format: aabbggrr
    styles = {
        'blue': 'ffff0000',  # blue
        'red':  'ff0000ff'   # red
    }

    for name, color_value in styles.items():
        # Line style
        style = ET.SubElement(document, 'Style', id=f'{name}Line')
        linestyle = ET.SubElement(style, 'LineStyle')
        ET.SubElement(linestyle, 'color').text = color_value
        ET.SubElement(linestyle, 'width').text = '3'

        # Hide default line labels
        labelstyle = ET.SubElement(style, 'LabelStyle')
        ET.SubElement(labelstyle, 'scale').text = '0'

    # Label (text-only) styles — no icon pin
    label_styles = {
        'blueLabel': 'ffff0000',
        'redLabel': 'ff0000ff'
    }

    for name, color_value in label_styles.items():
        style = ET.SubElement(document, 'Style', id=name)
        iconstyle = ET.SubElement(style, 'IconStyle')
        ET.SubElement(iconstyle, 'scale').text = '0'  # hide the pushpin
        ET.SubElement(iconstyle, 'Icon')  # empty <Icon> prevents default icon
        labelstyle = ET.SubElement(style, 'LabelStyle')
        ET.SubElement(labelstyle, 'color').text = color_value
        ET.SubElement(labelstyle, 'scale').text = '1.0'

    # ---------- Floating Legend Overlay ----------
    screen_overlay = ET.SubElement(document, 'ScreenOverlay')
    ET.SubElement(screen_overlay, 'name').text = "Legend"
    icon = ET.SubElement(screen_overlay, 'Icon')
    ET.SubElement(icon, 'href').text = ''  # no image; text-only legend
    overlay_desc = ET.SubElement(screen_overlay, 'description')
    overlay_desc.text = (
        "<div style='background-color:rgba(255,255,255,0.8); "
        "border:1px solid #000; padding:5px; font-size:12px;'>"
        "<b>Legend</b><br>"
        "<font color='blue'>Blue</font> = North / East<br>"
        "<font color='red'>Red</font> = South / West<br>"
        "<i>Lines offset by ~10 ft for visibility.<br>"
        "Labels offset by 50–500 ft depending on direction.<br>"
        "Label icons hidden (text only).</i>"
        "</div>"
    )
    # Position top-left corner
    ET.SubElement(screen_overlay, 'overlayXY', x="0", y="1", xunits="fraction", yunits="fraction")
    ET.SubElement(screen_overlay, 'screenXY', x="0.05", y="0.95", xunits="fraction", yunits="fraction")
    ET.SubElement(screen_overlay, 'size', x="0", y="0", xunits="pixels", yunits="pixels")

    # ---------- Read CSV ----------
    with open(csv_path, newline='', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            seg_id = row.get('Segment ID', '').strip()
            direction = row.get('Direction', '').strip().upper()
            try:
                start_lat = float(row.get('Start Latitude', ''))
                end_lat = float(row.get('End Latitude', ''))
                start_lon = float(row.get('Start Longitude', ''))
                end_lon = float(row.get('End Longitude', ''))
            except ValueError:
                continue

            # ---------- Apply 10 ft line offset ----------
            if direction == 'N':
                start_lon += lon_offset
                end_lon += lon_offset
            elif direction == 'S':
                start_lon -= lon_offset
                end_lon -= lon_offset
            elif direction == 'E':
                start_lat -= lat_offset
                end_lat -= lat_offset
            elif direction == 'W':
                start_lat += lat_offset
                end_lat += lat_offset

            # ---------- Choose styles ----------
            if direction in ['N', 'E']:
                line_style = '#blueLine'
                label_style = '#blueLabel'
            elif direction in ['S', 'W']:
                line_style = '#redLine'
                label_style = '#redLabel'
            else:
                line_style = '#blueLine'
                label_style = '#blueLabel'

            # ---------- Main Line Placemark ----------
            placemark = ET.SubElement(document, 'Placemark')
            ET.SubElement(placemark, 'name').text = f"{seg_id}, {direction}"
            ET.SubElement(placemark, 'description').text = (
                f"Segment ID: {seg_id}<br>Direction: {direction}"
            )
            ET.SubElement(placemark, 'styleUrl').text = line_style
            linestring = ET.SubElement(placemark, 'LineString')
            ET.SubElement(linestring, 'tessellate').text = '1'
            ET.SubElement(linestring, 'coordinates').text = f"{start_lon},{start_lat},0 {end_lon},{end_lat},0"

            # ---------- Label Placemark ----------
            if label_segments:
                mid_lat = (start_lat + end_lat) / 2
                mid_lon = (start_lon + end_lon) / 2

                # Directional label offsets
                if direction == 'E':
                    mid_lat -= 100 * lat_per_ft
                elif direction == 'W':
                    mid_lat += 100 * lat_per_ft
                elif direction == 'N':
                    mid_lon += 50 * lon_per_ft
                elif direction == 'S':
                    mid_lon += 50 * lon_per_ft

                label_placemark = ET.SubElement(document, 'Placemark')
                ET.SubElement(label_placemark, 'name').text = f"{seg_id} ({direction})"
                ET.SubElement(label_placemark, 'styleUrl').text = label_style
                point = ET.SubElement(label_placemark, 'Point')
                ET.SubElement(point, 'coordinates').text = f"{mid_lon},{mid_lat},0"

    # ---------- Write KML ----------
    tree = ET.ElementTree(kml)
    tree.write(kml_path, encoding='utf-8', xml_declaration=True)
    print(f"KML file created: {kml_path}")

# Example:
# csv_to_kml("segments.csv", "segments.kml", label_segments=True)


In [2]:
import csv
import xml.etree.ElementTree as ET

def csv_to_kml(csv_path, kml_path, label_segments=False, base_dir=None):
    """
    Convert a CSV with segment data to a KML file.

    Args:
        csv_path (str): Path to input CSV.
        kml_path (str): Path to output KML.
        label_segments (bool): If True, segment names are always visible (not just on hover).
    """

    # Approximate degree offsets for 10 ft at 44°N
    lat_offset_10ft = 10 / 364000
    lon_offset_10ft = 10 / 262000

    # Create root KML element
    kml = ET.Element('kml', xmlns='http://www.opengis.net/kml/2.2')
    document = ET.SubElement(kml, 'Document')

    # ---------- Styles ----------
    styles = {
        'blue': 'ffff0000',  # blue
        'red':  'ff0000ff'   # red
    }

    for name, color_value in styles.items():
        style = ET.SubElement(document, 'Style', id=f'{name}Line')

        # Line style
        linestyle = ET.SubElement(style, 'LineStyle')
        ET.SubElement(linestyle, 'color').text = color_value
        ET.SubElement(linestyle, 'width').text = '3'

        # Label style
        labelstyle = ET.SubElement(style, 'LabelStyle')
        ET.SubElement(labelstyle, 'scale').text = '1.0' if label_segments else '0'

        # Icon style (pins hidden)
        iconstyle = ET.SubElement(style, 'IconStyle')
        ET.SubElement(iconstyle, 'scale').text = '0'
        icon = ET.SubElement(iconstyle, 'Icon')
        ET.SubElement(icon, 'href').text = ''  # blank icon

    # ---------- Legend ----------
    legend = ET.SubElement(document, 'Placemark')
    ET.SubElement(legend, 'name').text = "Legend"
    ET.SubElement(legend, 'description').text = (
        "<b>Line Colors:</b><br>"
        "<font color='blue'>Blue</font> = North / East<br>"
        "<font color='red'>Red</font> = South / West<br>"
        "<br><i>Lines offset by ~10 ft for visibility.</i>"
    )

    # ---------- Read CSV ----------
    with open(csv_path, newline='', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            seg_id = row.get('Segment ID', '').strip()
            direction = row.get('Direction', '').strip().upper()

            try:
                start_lat = float(row.get('Start Latitude', ''))
                end_lat = float(row.get('End Latitude', ''))
                start_lon = float(row.get('Start Longitude', ''))
                end_lon = float(row.get('End Longitude', ''))
            except ValueError:
                continue  # skip invalid rows

            # Apply 10 ft offset for visibility
            if direction == 'N':
                start_lon += lon_offset_10ft
                end_lon += lon_offset_10ft
            elif direction == 'S':
                start_lon -= lon_offset_10ft
                end_lon -= lon_offset_10ft
            elif direction == 'E':
                start_lat -= lat_offset_10ft
                end_lat -= lat_offset_10ft
            elif direction == 'W':
                start_lat += lat_offset_10ft
                end_lat += lat_offset_10ft

            # Choose style
            if direction in ['N', 'E']:
                style_choice = '#blueLine'
            elif direction in ['S', 'W']:
                style_choice = '#redLine'
            else:
                style_choice = '#blueLine'

            # Center of line
            center_lat = (start_lat + end_lat) / 2
            center_lon = (start_lon + end_lon) / 2

            # Offset label position by direction
            if direction == 'E':
                label_lat = center_lat - (100 / 364000)
                label_lon = center_lon
            elif direction == 'W':
                label_lat = center_lat + (100 / 364000)
                label_lon = center_lon
            elif direction == 'N':
                label_lat = center_lat
                label_lon = center_lon + (50 / 262000)
            elif direction == 'S':
                label_lat = center_lat
                label_lon = center_lon + (50 / 262000)
            else:
                label_lat, label_lon = center_lat, center_lon

            # ---------- Placemark with MultiGeometry ----------
            placemark = ET.SubElement(document, 'Placemark')
            ET.SubElement(placemark, 'name').text = f"{seg_id}, {direction}"
            
            placemark_desc = f"Segment ID: {seg_id}<br>Direction: {direction}"
            if base_dir:
                scatter_link = f"file:///{base_dir}/{seg_id}_speed_scatter.html"
                placemark_desc += f"<br><a href='{scatter_link}'>Speed Scatter Plot</a>"
                summary_link = f"file:///{base_dir}/{seg_id}_daily_summary.html"
                placemark_desc += f"<br><a href='{summary_link}'>Segment Summary</a>"


            ET.SubElement(placemark, 'description').text = (
                f"{placemark_desc}"
            )
            ET.SubElement(placemark, 'styleUrl').text = style_choice

            multigeom = ET.SubElement(placemark, 'MultiGeometry')

            # LineString
            linestring = ET.SubElement(multigeom, 'LineString')
            ET.SubElement(linestring, 'tessellate').text = '1'
            ET.SubElement(linestring, 'coordinates').text = (
                f"{start_lon},{start_lat},0 {end_lon},{end_lat},0"
            )

            # Label Point (hidden pin)
            point = ET.SubElement(multigeom, 'Point')
            ET.SubElement(point, 'coordinates').text = (
                f"{label_lon},{label_lat},0"
            )

    # ---------- Write KML ----------
    tree = ET.ElementTree(kml)
    tree.write(kml_path, encoding='utf-8', xml_declaration=True)
    print(f"KML file created: {kml_path}")


In [4]:
csv_name = 'metadata.csv'
kml_name = 'segments.kml'
file_path = 'c:/Users/rhansen/Documents/Python/Inrix/sh-69/'
label_segments = True

csv_to_kml(file_path + csv_name, file_path + kml_name, label_segments=label_segments, base_dir=file_path)

KML file created: c:/Users/rhansen/Documents/Python/Inrix/sh-69/segments.kml
